# Intelligent Agents: Reflex-Based Agents for GridHunt

Student Name: [Add your name]

I have used the following AI tools: [list tools]

I understand that my submission needs to be my own work: [your initials]

## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: Undergrads 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and the hunter and the monster move on this grid. A hunter catches the monster if she/he moves on the same square the monster currently occupies. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunter are randomly placed in the arena environment. The hunter and the monster
    can move around, but cannot leave the arena.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena but only gets the monster location when it stops and listens.

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [1]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_agent_function, 
                      monster_agent_function, 
                      n = 30, max_steps = 100, 
                      visualize = False, animation = False):
    """
    Simulate an arena where a hunter agent tries to catch a monster agent.

    Parameters:
    hunter_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_pos = np.random.randint(0, n-1, size=2)
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step+1}: Hunter is at '{hunter_pos}'; Monster is at '{monster_pos}'")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_pos[0], hunter_pos[1]] = 'H'
        if np.array_equal(hunter_pos, monster_pos):
            arena[monster_pos[0], monster_pos[1]] = '*'
        print("\n".join("".join(row) for row in arena))

    # run the environment
    hunter_action = None  # Initialize hunter_action to None for the first step
    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_pos, monster_pos):
            if visualize:
                print(f"Hunter caught the monster in {step} steps!")
            return step
        
        # Get next move from monster agent function
        monster_action = monster_agent_function(hunter_pos, monster_pos)
        
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function. The monster's position is only provided 
        # if the hunter chose 'listen' as the previous action.
        if hunter_action == 'listen':
            monster_pos_instrument = monster_pos
        else: 
            monster_pos_instrument = None
        
        hunter_action = hunter_agent_function(hunter_pos, monster_pos_instrument)
        
        if hunter_action == 'listen':
            pass
        elif hunter_action == 'teleport':
            hunter_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_pos = move(hunter_action, hunter_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter chose action '{hunter_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Hunter failed to catch the monster in {max_steps} steps.")
    
    return np.nan

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [2]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(hunter_pos, monster_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [3]:
actions = ["north", "east", "west", "south", "teleport", "listen"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [4]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [5]:
simple_randomized_hunter_agent_function([0,0], None)

np.str_('north')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [6]:
arena_environment(simple_randomized_hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=5, max_steps=10, visualize=True, animation=False)

Step 1: Hunter is at '[0 2]'; Monster is at '[0 2]'
..*..
.....
.....
.....
.....
Hunter caught the monster in 0 steps!


0

**Note for VS Code:** View as scrollable element to see the complete output.

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $10 \times 10$ arena.  

In [7]:
steps = [arena_environment(simple_randomized_hunter_agent_function, monster_agent_function_simple, n=10, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[nan, 31, 10, 68, nan, 57, 15, 18, nan, nan, 0, nan, nan, nan, 81, nan, 0, 61, nan, 44, nan, nan, nan, 34, 40, 82, 1, 40, 14, 18, 71, nan, nan, nan, 94, nan, 86, nan, 66, 50, nan, nan, nan, nan, 9, nan, 52, 26, nan, 79, 14, 35, nan, nan, nan, nan, nan, nan, nan, nan, nan, 86, 56, nan, nan, 25, 67, 94, 2, nan, nan, nan, 90, 83, 94, nan, 92, 58, 55, nan, 14, 55, nan, nan, 39, nan, nan, 33, 25, nan, 96, nan, nan, 97, 56, nan, nan, 36, 31, 64]


We just got a list with the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results. `nan` means that the hunter was not able to catch the monster. How many times did that happen? To answer how well the hunter did when it caught the monster, we just average the numbers that are not `nan`. 

In [8]:
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 48 times.
Average steps to catch the monster over 52 successful runs: 48.92


This in not a very good agent. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and make it into a model-based reflex agent. It should use listen to get the monster's location and then move towards the monster. To do that it needs to remember the monster's location. This memory makes it model-based. Here is HOWTO: How to Store State Information in the Agent.

In [9]:
# here goes your agent implementation
def hunter_agent_function(hunter_location, monster_location):
    if monster_location is None:
            return 'listen'
    
    if monster_location is not None:
        if hunter_location[0] < monster_location[0]:
            return 'south'
        if hunter_location[0] > monster_location[0]:
            return 'north'
        if hunter_location[1] < monster_location[1]:
            return 'east'
        if hunter_location[1] > monster_location[1]:
            return 'west'
        else :
            return 'listen'
    

# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [10]:
# Example run of new hunter agent function
arena_environment(hunter_agent_function, 
                  monster_agent_function_simple, 
                  n=30, max_steps=100, visualize=True, animation=False)

Step 1: Hunter is at '[ 1 12]'; Monster is at '[ 4 27]'
..............................
............H.................
..............................
..............................
...........................M..
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
..............................
Hunter chose a

48

In [11]:
# Experiment code (10x10 grid, 100 runs, max 10 steps)
steps = [arena_environment(hunter_agent_function, 
                           monster_agent_function_simple, 
                           n=10, max_steps=30, visualize=False) for _ in range(100)] 
print(steps)

# Summary of results
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

[18, 28, 2, nan, nan, nan, 16, 18, 9, 4, nan, 8, 6, nan, 7, 18, 14, 2, 28, 5, 6, 27, 3, 9, 9, 20, 16, 18, 15, 8, 22, 10, 16, 10, 4, 12, 6, 12, 5, nan, 18, 4, 13, 14, 26, 25, 2, 14, 4, 16, 8, 2, 14, 22, 14, 14, 10, 4, 28, nan, nan, 8, nan, 22, nan, 20, 14, 10, 1, 4, 4, 7, 10, 22, 4, 8, 20, 8, 24, 26, 14, 20, 28, 6, nan, 10, 16, nan, 12, 22, 10, nan, 7, 12, 12, 21, 10, 29, 4, 4]
Hunter failed 13 times.
Average steps to catch the monster over 87 successful runs: 12.78


In [12]:
# Experiment code (30x30 grid, 100 runs, max 100 steps)
steps = [arena_environment(hunter_agent_function, 
                           monster_agent_function_simple, 
                           n=30, max_steps=90, visualize=False) for _ in range(100)] 
print(steps)

[56, nan, 78, 46, nan, 27, 20, 61, 42, 18, 48, 68, 72, 30, 50, 7, 70, 32, 45, 44, 40, 35, 82, 44, 8, 44, 62, 58, 40, 42, 76, nan, 88, 73, 68, 26, nan, 27, 64, 72, 86, 44, 13, 16, 28, 16, 48, 70, 52, 71, 88, 41, 22, 38, 43, 30, 40, 62, 16, 53, 33, 20, 10, 66, nan, nan, 36, 9, 43, nan, 2, 62, 8, 25, 57, 12, 32, nan, 22, 82, 82, 26, 42, 18, 17, 25, 31, 26, 48, 32, 45, nan, 58, 88, nan, 47, 38, nan, 68, 70]


In [13]:
# Summary of results
print (f"Hunter failed {np.sum(np.isnan(steps))} times.")
print (f"Average steps to catch the monster over {np.sum(~np.isnan(steps))} successful runs: {round(np.nanmean(steps), 2)}")

Hunter failed 11 times.
Average steps to catch the monster over 89 successful runs: 44.04


## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

### Discussion
1. What is your hunter's final strategy to choose actions. Why does it work well?
The final strategy of my hunter agent is to move directly toward the monster's current location. At the beginning, if the hunter does not know the monster position, it chooses the listen action to obtain the monster's location.
After receiving the monster position, the agent compares the row and column coordinates of the hunter and the monster:
- If the hunter is below the monster, it moves north.
- If the hunter is above the monster, it moves south.
- If the hunter is on the left side of the monster, it moves east.
- If the hunter is on the right side of the monster, it moves west.
This strategy works well because the hunter always chooses an action that reduces the distance between itself and the monster. Instead of moving randomly, the agent follows a direct path to the target, which significantly increases the probability of catching the monster in fewer steps.

2. Do you use teleportation. Why and in what situation? Why not?
My hunter agent does not use the teleport action.
Teleportation moves the hunter to a random position in the arena. Although this action can be useful when the hunter has no information about the monster location or when the hunter is very far away, it can also make the situation worse because the new position may be farther from the monster.
Since my agent uses the listen action to obtain the monster location and then moves directly toward it, teleportation is unnecessary. Moving step-by-step toward the monster is more reliable because every action reduces the distance instead of introducing randomness.

3. How does the arena size affect your hunter agent's performance?
The size of the arena affects the number of steps required to catch the monster. In a smaller arena, such as 10×10, the maximum distance between the hunter and monster is limited, so the agent can usually reach the monster faster.
In a larger arena, such as 30×30, the hunter and monster can start much farther apart. Therefore, the agent needs more movement steps before reaching the target.
However, increasing the arena size does not significantly reduce the success rate of this agent because the strategy does not depend on random exploration. The hunter always knows the monster direction after listening and follows the shortest path toward it. The main impact of a larger arena is the increase in average catching steps.

The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

### Discussion
The monster agent can be improved by using a strategy based on the hunter's position. Instead of moving randomly, the monster can detect when the hunter is approaching and select movements that increase the distance between them. This allows the monster to escape more effectively, making it harder for the hunter to catch it.

# More Work (Optional)

Here are some ideas:

* Implement a better monster agent function and perform experiments
* Change the environment so multiple hunters can hunt the monster.
* Give the hunter a new action that shoots an arrow in a specific direction. 
  The arrow can go a maximum of 5 squares. If the hunter hits the monster, then it wins.